In [132]:
from sympy import *

In [133]:
divVs, divqD, Pt0, Pf0, phi0 = symbols('divVs, divqD, Pt0, Pf0, Φ0')
K_s, K_f, K_phi, eta_phi, dQdp, gamma = symbols('Ks, Kf, KΦ, ηΦ, ∂Q∂p, λ̇')
Pt, Pf, dt = symbols('Pt, Pf, Δt')
phi = symbols('Φ')

In [134]:
elastic = 1
viscous = 1
plastic = 1
single_phase = 0

In [135]:
dPtdt  = (Pt-Pt0)/dt
dPfdt  = (Pf-Pf0)/dt
dphidt = elastic/K_phi*(dPfdt-dPtdt) + viscous/eta_phi*(Pf - Pt)

if single_phase==1:
    dphidt = 0
    phi = 0

# dlnrhosdt = elastic * 1/K_s * (dPtdt - phi*dPfdt)/(1-phi) 
# Ps     = (Pt - phi*Pf)/(1-phi) 
dPsdt = (dPtdt - phi*dPfdt) /(1-phi)
# dPsdt = ((Pt - phi*Pf)/(1-phi) - (Pt0 - phi0*Pf0)/(1-phi0))/dt
# dPsdt     = dphidt*(Pt - Pf*phi)/(1-phi)**2 + (dPtdt - phi*dPfdt - Pf*dphidt) / (1 - phi)
dlnrhosdt = elastic * 1/K_s * ( dPsdt ) 
fpt       = dlnrhosdt - dphidt/(1-phi) + divVs
fpt       = fpt.simplify()

dlnrhofdt = dPfdt/K_f
fpf       = elastic * phi*dlnrhofdt + dphidt + phi*divVs + divqD
fpf       = fpf.simplify()

fphi = phi    - (phi0 + dphidt*dt)

display(fpt)
display(fpf)

(Ks*KΦ*divVs*Δt*ηΦ*(Φ - 1) + Ks*(KΦ*Δt*(Pf - Pt) + ηΦ*(Pf - Pf0 - Pt + Pt0)) + KΦ*ηΦ*(-Pt + Pt0 + Φ*(Pf - Pf0)))/(Ks*KΦ*Δt*ηΦ*(Φ - 1))

(Kf*KΦ*Δt*ηΦ*(divVs*Φ + divqD) + Kf*KΦ*Δt*(Pf - Pt) + Kf*ηΦ*(Pf - Pf0 - Pt + Pt0) + KΦ*Φ*ηΦ*(Pf - Pf0))/(Kf*KΦ*Δt*ηΦ)

In [136]:
P_trial = solve([fpt, fpf], [Pt, Pf])

In [137]:
julia_code(P_trial[Pt].simplify())

'(-Kf .* Ks .* KΦ .* divVs .* Δt .^ 2 - Kf .* Ks .* KΦ .* divqD .* Δt .^ 2 - Kf .* Ks .* divVs .* Δt .* ηΦ - Kf .* Ks .* divqD .* Δt .* ηΦ - Kf .* KΦ .* Pf0 .* Δt .* Φ + Kf .* KΦ .* Pt0 .* Δt - Kf .* KΦ .* divVs .* Δt .* Φ .^ 2 .* ηΦ - Kf .* KΦ .* divqD .* Δt .* Φ .* ηΦ - Kf .* Pt0 .* Φ .* ηΦ + Kf .* Pt0 .* ηΦ + Ks .* KΦ .* Pf0 .* Δt .* Φ + Ks .* KΦ .* divVs .* Δt .* Φ .^ 2 .* ηΦ - Ks .* KΦ .* divVs .* Δt .* Φ .* ηΦ + Ks .* Pt0 .* Φ .* ηΦ + KΦ .* Pt0 .* Φ .* ηΦ) ./ (-Kf .* KΦ .* Δt .* Φ + Kf .* KΦ .* Δt - Kf .* Φ .* ηΦ + Kf .* ηΦ + Ks .* KΦ .* Δt .* Φ + Ks .* Φ .* ηΦ + KΦ .* Φ .* ηΦ)'

In [138]:
if single_phase==0:
    julia_code(P_trial[Pf].simplify())

In [139]:
if single_phase==0:
    julia_code(solve(fphi, phi)[0] .simplify())

In [140]:
dPt, dPf = symbols('ΔPt, ΔPf')

In [141]:
Pt1 = Pt+0*dPt
Pf1 = Pf+0*dPf
dPtdt  = (Pt1-Pt0)/dt
dPfdt  = (Pf1-Pf0)/dt
# phi    = phi0 + dphidt*dt

if single_phase==1:
    dphidt = -plastic*gamma*dQdp
else:
    # to be addressed (Thomsen)
    dphidt = elastic*(dPfdt-dPtdt)/K_phi + viscous*(Pf1 - Pt1)/eta_phi - plastic*gamma*dQdp
    
# dlnrhosdt = elastic * 1/K_s * (dPtdt - phi*dPfdt)/(1-phi) 
# Ps     = (Pt - phi*Pf)/(1-phi) 
dPsdt = (dPtdt - phi*dPfdt) /(1-phi)
# dPsdt = ((Pt - phi*Pf)/(1-phi) - (Pt0 - phi0*Pf0)/(1-phi0))/dt
# dPsdt = dphidt*(Pt - Pf*phi)/(1-phi)**2 + (dPtdt - phi*dPfdt - Pf*dphidt) / (1 - phi)
dlnrhosdt = elastic * 1/K_s * ( dPsdt ) 

fpt = dlnrhosdt - dphidt/(1-phi) + divVs
fpt = fpt.simplify()

dlnrhofdt = dPfdt/K_f 
fpf  = elastic*phi*dlnrhofdt + dphidt + phi*divVs + divqD
fpf = fpf.simplify()

if single_phase==1:     
    P_corr = solve([fpt], [Pt])
else:
    P_corr = solve([fpt, fpf], [Pt, Pf])
P_corr

{Pf: (Kf*Ks*KΦ*divVs*Δt**2 + Kf*Ks*KΦ*divqD*Δt**2 + Kf*Ks*divVs*Δt*ηΦ + Kf*Ks*divqD*Δt*ηΦ + Kf*KΦ*Pf0*Δt*Φ - Kf*KΦ*Pt0*Δt + Kf*KΦ*divVs*Δt*Φ*ηΦ + Kf*KΦ*divqD*Δt*ηΦ - Kf*KΦ*Δt*ηΦ*λ̇*∂Q∂p + Kf*Pf0*Φ*ηΦ - Kf*Pf0*ηΦ - Ks*KΦ*Pf0*Δt*Φ - Ks*Pf0*Φ*ηΦ - KΦ*Pf0*Φ*ηΦ)/(Kf*KΦ*Δt*Φ - Kf*KΦ*Δt + Kf*Φ*ηΦ - Kf*ηΦ - Ks*KΦ*Δt*Φ - Ks*Φ*ηΦ - KΦ*Φ*ηΦ),
 Pt: (Kf*Ks*KΦ*divVs*Δt**2 + Kf*Ks*KΦ*divqD*Δt**2 + Kf*Ks*divVs*Δt*ηΦ + Kf*Ks*divqD*Δt*ηΦ + Kf*KΦ*Pf0*Δt*Φ - Kf*KΦ*Pt0*Δt + Kf*KΦ*divVs*Δt*Φ**2*ηΦ + Kf*KΦ*divqD*Δt*Φ*ηΦ - Kf*KΦ*Δt*Φ*ηΦ*λ̇*∂Q∂p + Kf*Pt0*Φ*ηΦ - Kf*Pt0*ηΦ - Ks*KΦ*Pf0*Δt*Φ - Ks*KΦ*divVs*Δt*Φ**2*ηΦ + Ks*KΦ*divVs*Δt*Φ*ηΦ + Ks*KΦ*Δt*Φ*ηΦ*λ̇*∂Q∂p - Ks*Pt0*Φ*ηΦ - KΦ*Pt0*Φ*ηΦ)/(Kf*KΦ*Δt*Φ - Kf*KΦ*Δt + Kf*Φ*ηΦ - Kf*ηΦ - Ks*KΦ*Δt*Φ - Ks*Φ*ηΦ - KΦ*Φ*ηΦ)}

In [142]:
P_trial[Pt]

(Kf*Ks*KΦ*divVs*Δt**2 + Kf*Ks*KΦ*divqD*Δt**2 + Kf*Ks*divVs*Δt*ηΦ + Kf*Ks*divqD*Δt*ηΦ + Kf*KΦ*Pf0*Δt*Φ - Kf*KΦ*Pt0*Δt + Kf*KΦ*divVs*Δt*Φ**2*ηΦ + Kf*KΦ*divqD*Δt*Φ*ηΦ + Kf*Pt0*Φ*ηΦ - Kf*Pt0*ηΦ - Ks*KΦ*Pf0*Δt*Φ - Ks*KΦ*divVs*Δt*Φ**2*ηΦ + Ks*KΦ*divVs*Δt*Φ*ηΦ - Ks*Pt0*Φ*ηΦ - KΦ*Pt0*Φ*ηΦ)/(Kf*KΦ*Δt*Φ - Kf*KΦ*Δt + Kf*Φ*ηΦ - Kf*ηΦ - Ks*KΦ*Δt*Φ - Ks*Φ*ηΦ - KΦ*Φ*ηΦ)

In [143]:
dPt = ((P_corr[Pt] - P_trial[Pt])).simplify()

if single_phase==0:
    dPf = ((P_corr[Pf] - P_trial[Pf])).simplify()
    julia_code(dPf)

display(dPf)
display(dPt)


Kf*KΦ*Δt*ηΦ*λ̇*∂Q∂p/(-Kf*KΦ*Δt*Φ + Kf*KΦ*Δt - Kf*Φ*ηΦ + Kf*ηΦ + Ks*KΦ*Δt*Φ + Ks*Φ*ηΦ + KΦ*Φ*ηΦ)

KΦ*Δt*Φ*ηΦ*λ̇*∂Q∂p*(Kf - Ks)/(-Kf*KΦ*Δt*Φ + Kf*KΦ*Δt - Kf*Φ*ηΦ + Kf*ηΦ + Ks*KΦ*Δt*Φ + Ks*Φ*ηΦ + KΦ*Φ*ηΦ)

In [144]:
print('ΔPt = ' +  julia_code(dPt))
print('ΔPf = ' +  julia_code(dPf))


ΔPt = KΦ .* Δt .* Φ .* ηΦ .* λ̇ .* ∂Q∂p .* (Kf - Ks) ./ (-Kf .* KΦ .* Δt .* Φ + Kf .* KΦ .* Δt - Kf .* Φ .* ηΦ + Kf .* ηΦ + Ks .* KΦ .* Δt .* Φ + Ks .* Φ .* ηΦ + KΦ .* Φ .* ηΦ)
ΔPf = Kf .* KΦ .* Δt .* ηΦ .* λ̇ .* ∂Q∂p ./ (-Kf .* KΦ .* Δt .* Φ + Kf .* KΦ .* Δt - Kf .* Φ .* ηΦ + Kf .* ηΦ + Ks .* KΦ .* Δt .* Φ + Ks .* Φ .* ηΦ + KΦ .* Φ .* ηΦ)


In [155]:
dPt, dPf = symbols('dPt, dPf')
Pt1 = dPt
Pf1 = dPf
dPtdt  = (Pt1-0*Pt0)/dt
dPfdt  = (Pf1-0*Pf0)/dt
# phi    = phi0 + dphidt*dt

if single_phase==1:
    dphidt = -plastic*gamma*dQdp
else:
    # to be addressed (Thomsen)
    dphidt = elastic*(dPfdt-dPtdt)/K_phi + viscous*(Pf1 - Pt1)/eta_phi - plastic*gamma*dQdp
    
# dlnrhosdt = elastic * 1/K_s * (dPtdt - phi*dPfdt)/(1-phi) 
# Ps     = (Pt - phi*Pf)/(1-phi) 
dPsdt = (dPtdt - phi*dPfdt) /(1-phi)
# dPsdt = ((Pt - phi*Pf)/(1-phi) - (Pt0 - phi0*Pf0)/(1-phi0))/dt
# dPsdt = dphidt*(Pt - Pf*phi)/(1-phi)**2 + (dPtdt - phi*dPfdt - Pf*dphidt) / (1 - phi)
dlnrhosdt = elastic * 1/K_s * ( dPsdt ) 

fpt = dlnrhosdt - dphidt/(1-phi) + divVs*0
fpt = fpt.simplify()

dlnrhofdt = dPfdt/K_f 
fpf  = elastic*phi*dlnrhofdt + dphidt + phi*divVs*0 + divqD*0
fpf = fpf.simplify()

if single_phase==1:     
    dP = solve([fpt], [dPt])
else:
    dP = solve([fpt, fpf], [dPt, dPf])

In [156]:
dPt = (dP[dPt]).simplify()

if single_phase==0:
    dPf = ((dP[dPf])).simplify()
    julia_code(dPf)

display(dPf)
display(dPt)

Kf*KΦ*Δt*ηΦ*λ̇*∂Q∂p/(-Kf*KΦ*Δt*Φ + Kf*KΦ*Δt - Kf*Φ*ηΦ + Kf*ηΦ + Ks*KΦ*Δt*Φ + Ks*Φ*ηΦ + KΦ*Φ*ηΦ)

KΦ*Δt*Φ*ηΦ*λ̇*∂Q∂p*(Kf - Ks)/(-Kf*KΦ*Δt*Φ + Kf*KΦ*Δt - Kf*Φ*ηΦ + Kf*ηΦ + Ks*KΦ*Δt*Φ + Ks*Φ*ηΦ + KΦ*Φ*ηΦ)

In [157]:
print('ΔPt = ' +  julia_code(dPt))
print('ΔPf = ' +  julia_code(dPf))

ΔPt = KΦ .* Δt .* Φ .* ηΦ .* λ̇ .* ∂Q∂p .* (Kf - Ks) ./ (-Kf .* KΦ .* Δt .* Φ + Kf .* KΦ .* Δt - Kf .* Φ .* ηΦ + Kf .* ηΦ + Ks .* KΦ .* Δt .* Φ + Ks .* Φ .* ηΦ + KΦ .* Φ .* ηΦ)
ΔPf = Kf .* KΦ .* Δt .* ηΦ .* λ̇ .* ∂Q∂p ./ (-Kf .* KΦ .* Δt .* Φ + Kf .* KΦ .* Δt - Kf .* Φ .* ηΦ + Kf .* ηΦ + Ks .* KΦ .* Δt .* Φ + Ks .* Φ .* ηΦ + KΦ .* Φ .* ηΦ)
